# coop_grid

> Phase 12 grids and frozen analysis plans: Track A (instructions and channels for Claude: `trackAGrid`, `trackAPilotGrid`, `TRACKA_PLAN`) and Track B (`RL_PLAN`, `evalSuite`); `coopHash`.

In [ ]:
#| default_exp coop_grid

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import hashlib
import json
from dataclasses import asdict

from explRL.clue_grid import BOARD, CLUE_SETTINGS, clueCell
from explRL.clue_rl import RLConfig
from explRL.clue_sft import SFTConfig
from explRL.llm_budget import PRICES
from explRL.llm_grid import HAIKU

MODELS = dict(base='Qwen/Qwen2.5-3B-Instruct', debug='Qwen/Qwen2.5-0.5B-Instruct')   # candidates; confirmed in T7, fixed in RL_PLAN
TRACKA = (('A1', 'hierarchical', dict(hint='follow')), ('A2', 'hierarchical', dict(passPooled=True)),
          ('A3', 'blackboard', dict(showOpen=True)), ('A3', 'hierarchical', dict(showOpen=True)))


def _world(opt, s=CLUE_SETTINGS):
    return {**{k: s[k] for k in BOARD}, **opt}


def trackAGrid(s=CLUE_SETTINGS):
    "Track A main: each (variant, channel) at N 4 and 8, seeds 900-904, 40 turns (paired with the Phase 11 cells)."
    return [clueCell('coopA', v, N, sd, s=s, channel=ch, world=_world(o, s))
            for v, ch, o in TRACKA for N in (4, 8) for sd in range(900, 905)]


def trackAPilotGrid(s=CLUE_SETTINGS):
    "Track A pilot: each (variant, channel) at N = 4, seed 1100, 30 turns."
    return [clueCell('coopAPilot', v, 4, 1100, s=s, channel=ch, world=_world(o, s), turns=30) for v, ch, o in TRACKA]


def trackAQuick():
    "The pilot cells at 3 turns, for the fake-client pipeline check."
    return [dict(c, cfg=dict(c['cfg'], turns=3)) for c in trackAPilotGrid()]


TRACKA_PLAN = dict(
    version=1, frozen=True, model=HAIKU, settings=CLUE_SETTINGS, variants=[list(x) for x in TRACKA], prices=PRICES,
    seeds=dict(pilot=[1100], main=[900, 904]), baseline='results/multiagent/clue/full (study clue, Phase 11)',
    analyses=dict(
        effect='per variant, channel, N: welfare minus the Phase 11 cell (same channel, N, seed), paired, bootstrap CI',
        gap='per coordinator variant and N: welfare minus the Phase 11 blackboard cell, paired on seed',
        mechanism='follow share (coordinator), shares per held clue, redundant shares, solo solves (from the turn logs)'),
    nboot=2000, ci=0.95, exploratory=True)


LOCAL = ('untrained', 'warm', 'individual', 'team', 'credit')      # the models evaluated (main boards: all five)
MAIN_SEEDS = list(range(900, 905)) + list(range(930, 940))
RL_PLAN = dict(
    version=1, frozen=True, models=MODELS, settings=CLUE_SETTINGS, schemes=['individual', 'team', 'credit'], prices=PRICES,
    sft={k: v for k, v in asdict(SFTConfig('', '')).items() if k not in ('model', 'out', 'device')},
    sftData=dict(seeds=[20000, 20099], sizes=[1, 2, 4], turns=20),
    warmCheck=dict(seeds=[1200, 1219], turns=30, solveRate=0.8, invalid=0.05),
    rl={**{k: v for k, v in asdict(RLConfig('', '')).items() if k not in ('model', 'out', 'device', 'scheme')},
        'gradCkpt': False,                    # the A100 has the memory (speed check: 50.3 GB of 80)
        'steps': 60},                         # speed check: 166 s per step, so 2.8 hours per run (P12-9)
    haiku=dict(model=HAIKU, memory='full', framing='own', effort=CLUE_SETTINGS['effort'], maxTokens=CLUE_SETTINGS['maxTokens']),
    seeds=dict(train='boardSeed0 + step x seedsPerStep + j', main=MAIN_SEEDS, generalisation=[900, 904], crossplay=[900, 904],
               liar=[900, 902]),
    analyses=dict(
        main='N = 4, blackboard, 40 turns, seeds 900-904 and 930-939: welfare and pooled-bound share of the five models',
        gain='each RL model minus the warm-start model, paired on board seed',
        schemes='credit - individual, team - individual, credit - team, paired on board seed',
        phase11='each model minus Haiku and minus the rule agents (saved Phase 11 rows), seeds 900-904, paired',
        information='shares per held clue, redundant shares, share-to-use time, guesses, solo share, wrong per solve',
        generalisation='N = 8 blackboard; N = 4 messages; N = 4 coordinator (seeds 900-904): minus the Phase 11 Haiku cells; follow share',
        crossplay='N = 4 blackboard: one model agent + three Haiku, one Haiku + three model agents: minus all-Haiku (Phase 11)',
        liar='N = 4, one scripted liar, blackboard and messages, seeds 900-902: honest welfare lost against the clean boards; '
             'blackboard-minus-messages sign (H3)',
        perRole='optional: credit reward, main boards: per-role adapters minus the shared adapter'),
    nboot=2000, ci=0.95, exploratory=True)


def evalCell(model, test, channel, N, seed, **cfg):
    "An evaluation cell (study coopEval, level '<model>|<test>', runner 'local'), 40 turns, notes memory, Phase 11 settings."
    return clueCell('coopEval', f'{model}|{test}', N, seed, 'local', channel=channel, model=f'local:{model}', memory='notes',
                    world=_world({}), **cfg)


def evalSuite(models=LOCAL, extra=()):
    "RL_PLAN's evaluation cells: main boards for every model (and `extra`, e.g. 'perRole'); the other tests without 'untrained'."
    cells = [evalCell(m, 'main', 'blackboard', 4, sd) for m in list(models) + list(extra) for sd in MAIN_SEEDS]
    for m in (x for x in models if x != 'untrained'):
        g = range(900, 905)
        cells += [evalCell(m, t, ch, N, sd) for t, ch, N in (('gen8', 'blackboard', 8), ('genMsg', 'messages', 4),
                                                              ('genLead', 'hierarchical', 4), ('cross1', 'blackboard', 4),
                                                              ('cross3', 'blackboard', 4)) for sd in g]
        cells += [evalCell(m, 'liar', ch, 4, sd, nLiar=1) for ch in ('blackboard', 'messages') for sd in range(900, 903)]
    return cells


def coopHash(plan):
    return hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest()[:12]

## Tests

In [ ]:
from explRL.coop_grid import *
from explRL.llm_grid import estimateGrid
from explRL.llm_runs import LLMConfig, cellRow
from explRL.bench_grid import rowKey

calls = lambda cells: sum(v['calls'] for v in estimateGrid(cells).values())
g, p = trackAGrid(), trackAPilotGrid()
assert len(g) == 4 * 2 * 5 and calls(g) == 9600 and len(p) == 4 and calls(p) == 480
assert 2.0 < sum(v['cost'] for v in estimateGrid(g).values()) < 2.6
keys = [rowKey(cellRow(c, LLMConfig.fromDict(c['cfg']), dict(episode=0))) for c in g + p]
assert len(set(keys)) == len(keys)
# options in the world, the Phase 11 board settings otherwise; A1 and A2 only on the coordinator
for c in g:
    w = c['cfg']['world']
    assert {k: w[k] for k in BOARD} == {k: CLUE_SETTINGS[k] for k in BOARD}
    assert (c['level'] in ('A1', 'A2')) <= (c['cfg']['channel'] == 'hierarchical')
assert {c['seed'] for c in g} == set(range(900, 905)) and all(c['cfg']['turns'] == 30 for c in p)
assert all(c['cfg']['turns'] == 3 for c in trackAQuick()) and len(trackAQuick()) == 4
# the evaluation suite: turns of local-model agents per model as in design-low-level section 8
turns = lambda cells, m: sum((c['N'] - c['cfg']['nLiar']) * c['cfg']['turns'] for c in cells if c['level'].startswith(m + '|'))
ev = evalSuite()
assert turns(ev, 'untrained') == 2400 and turns(ev, 'warm') == 2400 + 3200 + 2 * 800 + 720
assert sum(c['N'] * 40 for c in ev if c['level'].endswith(('cross1', 'cross3'))) == 4 * 2 * 5 * 160
assert all(c['runner'] == 'local' and c['cfg']['turns'] == 40 and c['cfg']['memory'] == 'notes' for c in ev)
assert len({rowKey(cellRow(c, LLMConfig.fromDict(c['cfg']), dict(episode=0))) for c in ev}) == len(ev)
assert len(evalSuite(extra=('perRole',))) == len(ev) + 15
assert RL_PLAN['rl']['turnFraction'] == 0.25 and RL_PLAN['rl']['gradCkpt'] is False and RL_PLAN['rl']['haltRun'] == 3
assert RL_PLAN['prices'] == TRACKA_PLAN['prices'] and RL_PLAN['rl']['steps'] == 60 and RL_PLAN['seeds']['main'] == MAIN_SEEDS and coopHash(RL_PLAN) != coopHash(TRACKA_PLAN)
assert len(coopHash(TRACKA_PLAN)) == 12 and coopHash({**TRACKA_PLAN, 'version': 2}) != coopHash(TRACKA_PLAN)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()